In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.dirname(__file__) if '__file__' in globals() else '.'))

from utils.triathlon_utils import (
    parse_fit, 
    clean_fit_data, 
    calculate_elapsed_time, 
    plot_race_anatomy_template, 
    plot_run_comparison,
    export_for_instagram,
    KLEUREN_PALETTE
)

from utils.config import DEFAULT_ATHLETE, DEFAULT_PATHS

In [0]:
fit_file_path = f"{DEFAULT_PATHS.fit_dir}/20260829_run-bike-run.fit"

# 1. Inlezen & Opschonen
df = parse_fit(fit_file_path)
df_clean = clean_fit_data(df)
df_active = calculate_elapsed_time(df_clean)

In [0]:
# 1. Bepaal de fietstijden op basis van snelheid (> 20 km/u)
bike_mask = df_active['speed_kmh'] > 20
bike_start = df_active[bike_mask]['elapsed_minutes'].min()
bike_eind = df_active[bike_mask]['elapsed_minutes'].max()

# Bepaal overgangspunten t1_start en t2_eind
t1_candidates = df_active[(df_active['elapsed_minutes'] < bike_start) & (df_active['speed_kmh'] < 3)]['elapsed_minutes']
t1_start = t1_candidates.min() if not t1_candidates.empty else bike_start - 2.0

# T2: Eerst het moment van afremmen/stilstaan in de wisselzone (< 3 km/u)
t2_stops = df_active[(df_active['elapsed_minutes'] >= bike_eind) & 
                     (df_active['elapsed_minutes'] <= bike_eind + 5) & 
                     (df_active['speed_kmh'] < 3)]['elapsed_minutes']

if not t2_stops.empty:
    laatste_stop = t2_stops.max()
    run2_candidates = df_active[(df_active['elapsed_minutes'] >= laatste_stop) & 
                                (df_active['speed_kmh'] > 8.5)]['elapsed_minutes']
    t2_eind = run2_candidates.min() if not run2_candidates.empty else bike_eind + 2.0
else:
    t2_eind = bike_eind + 2.0

# 2. Definieer fase-indeling met kleuren
fases = [
    {'naam': 'RUN 1', 'start': 0, 'eind': t1_start, 'kleur': KLEUREN_PALETTE['speed']},
    {'naam': 'T1',    'start': t1_start, 'eind': bike_start, 'kleur': KLEUREN_PALETTE['muted']},
    {'naam': 'BIKE',  'start': bike_start, 'eind': bike_eind, 'kleur': KLEUREN_PALETTE['bike']},
    {'naam': 'T2',    'start': bike_eind, 'eind': t2_eind, 'kleur': KLEUREN_PALETTE['muted']},
    {'naam': 'RUN 2', 'start': t2_eind, 'eind': df_active['elapsed_minutes'].max(), 'kleur': KLEUREN_PALETTE['hr']}
]

# 3. Genereer en exporteer de grafiek
fig, ax1, ax2 = plot_race_anatomy_template(df_active, fases)
export_for_instagram(fig, filename="20260829_race_opbouw", format_type='portrait')

In [0]:
fig2 = plot_run_comparison(df_active, bike_start, bike_eind)
export_for_instagram(fig2, filename="20260829_run1_vs_run2", format_type='portrait')